## Three questions
1. Did women survive more than men?
2. Did ticket class (pclass) matter for survival?
3. Was age related to survival?

In [1]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")
print("Rows and columns:", df.shape)
df.head()

Rows and columns: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## Understanding the columns
- **survived**: 1 = survived, 0 = did not survive
- **pclass**: ticket class (1 = first, 2 = second, 3 = third)
- **sex**: male or female
- **age**: age in years (some values missing)
- **sibsp**: number of siblings or spouses aboard
- **parch**: number of parents or children aboard
- **fare**: ticket price
- **embarked**: boarding port code (C = Cherbourg, Q = Queenstown, S = Southampton)
- **class**: same information as pclass, stored as text
- **who**: man, woman or child
- **adult_male**: True if an adult male
- **deck**: cabin deck letter (mostly missing)
- **embark_town**: boarding port name, same information as embarked
- **alive**: text version of survived
- **alone**: True if travelling without family

In [2]:
print(df.info())
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB
None


,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [3]:
df.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

## Handling missing values
Before cleaning, `age` had 177 missing values, `deck` had 688, and `embarked` and `embark_town` had 2 each.

- **Age**: filled with the median age of passengers in the same class and sex. Dropping 177 rows would lose too much data, and passengers in the same group had similar ages.
- **Embarked and embark_town**: filled with the most common port. Only 2 values are missing, so this is a safe estimate.
- **Deck**: dropped, since about 77% of its values are empty and it can't tell us much.

In [4]:
# 1. Age: fill each gap with the median age of passengers in the same class and sex
df["age"] = df["age"].fillna(df.groupby(["pclass", "sex"])["age"].transform("median"))

# 2. Embarked: fill the 2 gaps with the most common port
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

# 3. Deck: drop the column, since most of it is empty
df = df.drop(columns=["deck"])

# 4. Check the result
print(df.isna().sum())

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    2
alive          0
alone          0
dtype: int64


In [5]:
# Fill the 2 missing embark_town values with the most common town
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])

# Check that no missing values are left
print("Missing values left:", df.isna().sum().sum())

# Count duplicate rows (identical passengers listed twice)
print("Duplicate rows:", df.duplicated().sum())

# Convert text columns that are really categories
for col in ["sex", "embarked", "embark_town", "who", "alive"]:
    df[col] = df[col].astype("category")

# Show the final column types
print(df.dtypes)

Missing values left: 0
Duplicate rows: 118
survived          int64
pclass            int64
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
embarked       category
class          category
who            category
adult_male         bool
embark_town    category
alive          category
alone              bool
dtype: object


## Duplicates
There are 118 duplicate rows. The dataset has no passenger ID, so identical rows may be different people. We keep them.

In [6]:
# Survival rate by sex (sorted)
print(df.groupby("sex", observed=True)["survived"].mean().sort_values(ascending=False).round(2))

# Survival rate by ticket class (sorted), with passenger counts
print(df.groupby("pclass")["survived"].agg(["mean", "count"]).sort_values("mean", ascending=False).round(2))

# Group ages into bands, then compare survival by band
df["age_band"] = pd.cut(df["age"], [0, 12, 18, 40, 60, 100],
                        labels=["Child", "Teen", "Adult", "Middle-aged", "Senior"])
print(df.groupby("age_band", observed=True)["survived"].mean().sort_values(ascending=False).round(2))

sex
female    0.74
male      0.19
Name: survived, dtype: float64
        mean  count
pclass             
1       0.63    216
2       0.47    184
3       0.24    491
age_band
Child          0.58
Teen           0.43
Middle-aged    0.39
Adult          0.36
Senior         0.23
Name: survived, dtype: float64


## Answers

1. **Did women survive more than men?** Yes. About 74% of women survived, compared with about 19% of men.
2. **Did ticket class matter?** Yes. Survival was about 63% in first class, 47% in second class and 24% in third class.
3. **Was age related to survival?** Yes. Children had the highest survival rate (about 58%) and seniors the lowest (about 23%). Survival generally falls with age.

These are associations in the data, not proof of cause. Class and sex may overlap, so the effects can't be fully separated here.